# Bragg peaks: all reflections

Column-sum Pseudo-Voigt lineouts for every indexed peak: (011), (10-1), (120), and (101) at 320–350 K, then (111) from 360 K.

- **Left:** $I(|Q|)$ for each temperature, offset vertically. Peak colors match the $Q$-dependence figure.
- **Right:** $|\mathbf{Q}_0|$ and $\xi$ vs $T$ for all peaks, same single-peak Pseudo-Voigt as the tracked (120) $\rightarrow$ (111) figure.

(120) and (111) use the same processed / native-ROI loading as that figure so the points match. The other three peaks are raw frames cropped to each XPCS ROI.


In [ ]:
import sys
sys.path.append('../src/')

from pathlib import Path
import json

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import median_filter
from scipy.optimize import brentq, curve_fit, minimize

try:
    import hdf5plugin  # noqa: F401
except ImportError:
    pass

from postprocess import (
    FOUR_PEAK_SCANS,
    REFLECTION_ORDER,
    SINGLE_PEAK_SCANS,
    TRACKED_SCANS,
    temperature_cmap,
    temperature_color,
    temperature_norm,
)
from xpcs import pseudo_voigt


In [ ]:
def apply_paper_style():
    helvetica = font_manager.FontProperties(family='Helvetica')
    font_manager.findfont(helvetica, fallback_to_default=False)
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Helvetica', 'Arial'],
        'font.size': 9,
        'axes.labelsize': 11,
        'axes.titlesize': 11,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 8,
        'legend.frameon': False,
        'axes.linewidth': 1.0,
        'lines.linewidth': 1.2,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.major.width': 1.0,
        'ytick.major.width': 1.0,
        'xtick.major.size': 4.5,
        'ytick.major.size': 4.5,
        'xtick.top': True,
        'ytick.right': True,
        'mathtext.fontset': 'custom',
        'mathtext.rm': 'Helvetica',
        'mathtext.it': 'Helvetica:italic',
        'mathtext.bf': 'Helvetica:bold',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.dpi': 600,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.03,
    })


apply_paper_style()

cmap = temperature_cmap()
norm = temperature_norm()
FIGSIZE = (11.0, 5.6)
red = '#e31a1c'
blue = '#263fc2'
SPT = (360.0, 373.0)
spt_cmap = LinearSegmentedColormap.from_list(
    'spt_warm',
    ['#fff9b0', '#ffd166', '#ff9f1c', '#e85d04'],
    N=256,
)

MARKERS = {
    '011': 'o',
    '10-1': 's',
    '120': 'D',
    '101': '^',
    '111': '*',
}
HKL_COLORS = {
    '011': '#2E5E8C',
    '10-1': '#2F8A6B',
    '120': '#C97A2B',
    '101': '#9E3B4E',
    '111': '#6B3D91',
}
HKL_LABELS = {
    '011': '(011)',
    '10-1': r'(10$\overline{1}$)',
    '120': '(120)',
    '101': '(101)',
    '111': '(111)',
}

L_FILE_MM = 2200.0
L_TRUE_MM = 2020.0
Q_REF_120 = 1.1429
SCHERRER_K = 0.9
NFRAMES_PROCESSED = 200
NFRAMES_RAW = 50
OFFSET = 1.25

config_path = Path('../configs/config_B10.json')
config = json.loads(config_path.read_text()) if config_path.exists() else {}
APS_BASE = Path(config['Base']) if config.get('Base') else Path('/Users/eriklamb/data/APS/8_ID_E/Na2B10H10')
DESK_PROCESSED = Path('/Users/eriklamb/Desktop/Na2b10h10/Processed')

ALL_PEAK_SCANS = tuple((T, scan, hkl) for T, scan in FOUR_PEAK_SCANS for hkl in REFLECTION_ORDER) + tuple(
    (T, scan, '111') for T, scan in SINGLE_PEAK_SCANS
)


In [ ]:
def qy_vertical_rows(rows, ver0, l_mm, wavelength, two_theta, pixel_mm):
    ver_angle = two_theta - np.degrees(np.arctan((rows - ver0) * pixel_mm / l_mm))
    return (4.0 * np.pi / wavelength) * np.sin(np.radians(ver_angle / 2.0))


def clean_mean_image(det, gap_frac=0.5, hot_sigma=5):
    med_time = np.median(det, axis=0)
    local_med = median_filter(med_time, size=7)
    local_mad = median_filter(np.abs(med_time - local_med), size=7)
    sigma = np.maximum(1.4826 * local_mad, 1.0)
    gap_mask = (med_time < gap_frac * local_med) & (local_med > 0)
    hot_mask = med_time > local_med + hot_sigma * sigma
    valid = ~(gap_mask | hot_mask)
    return det.mean(axis=0) * valid, valid


def infer_crop_v0(qy_ref, nrows, wavelength, two_theta, pixel_mm, ver0):
    def residual(params):
        rows = params[0] + np.arange(nrows)
        q_calc = qy_vertical_rows(rows, ver0, L_FILE_MM, wavelength, two_theta, pixel_mm)
        return np.mean((q_calc - qy_ref) ** 2)

    best = None
    for x0 in (0, 200, 400, 600, 700, 800, 1000):
        result = minimize(
            residual, [float(x0)], method='Nelder-Mead',
            options={'xatol': 1e-4, 'fatol': 1e-16, 'maxiter': 2000},
        )
        if best is None or result.fun < best.fun:
            best = result
    return float(best.x[0])


def find_processed(T, scan):
    desk = sorted(DESK_PROCESSED.glob(f'particleA7_temp{T}K_scan{scan}_*.h5'))
    if desk:
        return desk[0]
    aps = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001' / 'processed'
    if not aps.exists():
        return None
    hits = sorted(aps.glob(f'particleA7_temp{T}K_scan{scan}_*.h5'))
    hits.sort(key=lambda p: (('NewMask' not in p.name), -p.stat().st_size))
    return hits[0] if hits else None


def find_result(T, scan, hkl):
    folder = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001' / 'results'
    hits = list(folder.glob(f'*({hkl})*.h5'))
    if not hits:
        raise FileNotFoundError(f'no ({hkl}) result for T={T} scan={scan}')
    return hits[0]


def _scalar(value):
    return float(np.asarray(value).reshape(-1)[0])


def load_processed(path, nframes=NFRAMES_PROCESSED):
    with h5py.File(path, 'r') as f:
        qy_file = np.asarray(f['pre_processing/Qy'][:, 0], dtype=float)
        ntot = f['data/det_corr'].shape[0]
        det = np.asarray(f['data/det_corr'][:min(nframes, ntot)], dtype=float)
        temp = _scalar(f['experimental_parameters/temperature'][()])
        wl = _scalar(f['experimental_parameters/wavelength'][()])
        tth = _scalar(f['experimental_parameters/ttheta'][()])
        pix = _scalar(f['experimental_parameters/pixel_size'][()])
        Y0 = _scalar(f['experimental_parameters/Y0'][()])
    intensity = clean_mean_image(det)[0].sum(axis=1)
    crop_v0 = infer_crop_v0(qy_file, len(intensity), wl, tth, pix, Y0)
    return {
        'temperature': temp, 'wl': wl, 'tth': tth, 'pix': pix, 'Y0': Y0,
        'crop_v0': crop_v0, 'intensity': intensity, 'path': Path(path), 'source': 'processed',
    }


def load_raw_roi(T, scan, hkl, nframes=NFRAMES_RAW):
    result_path = find_result(T, scan, hkl)
    with h5py.File(result_path, 'r') as f:
        roi = np.asarray(f['peaks/peak_0/roi'][()]).astype(int)
        seed = np.asarray(f['peaks/peak_0/seed'][()], dtype=float)
        popt = np.asarray(f['peaks/peak_0/popt'][()], dtype=float).ravel()
        wl = _scalar(f['experimental_parameters/wavelength'][()])
        tth = _scalar(f['experimental_parameters/ttheta'][()])
        pix = _scalar(f['experimental_parameters/pixel_size'][()])
        Y0 = _scalar(f['experimental_parameters/Y0'][()])
    r0, r1, c0, c1 = [int(x) for x in roi.tolist()]
    raw = next((APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001').glob('*_001.h5'))
    with h5py.File(raw, 'r') as f:
        det = np.asarray(f['entry/data/data'][:nframes, r0:r1, c0:c1], dtype=float)
    intensity = clean_mean_image(det)[0].sum(axis=1)
    peak_row = float(popt[2]) if popt.size >= 3 else float(seed[1])
    return {
        'temperature': float(T), 'wl': wl, 'tth': tth, 'pix': pix, 'Y0': Y0,
        'crop_v0': float(r0), 'intensity': intensity, 'path': raw, 'source': 'raw',
        'roi': (r0, r1, c0, c1), 'seed_row': float(seed[1]), 'peak_row': peak_row,
    }


def locate_shoulder_main(q, y):
    i_main = int(np.argmax(y))
    low = y.copy()
    low[(q >= q[i_main] - 0.002) | (q < q[i_main] - 0.03)] = 0
    i_sh = int(np.argmax(low)) if low.max() > 0 else max(0, i_main - 8)
    if q[i_sh] >= q[i_main]:
        i_sh = max(0, i_main - 8)
    return i_sh, i_main


def fit_brightest(q, intensity, shoulder_mode=False, q_guess=None):
    y = intensity / intensity.max()
    i_sh, i_main = locate_shoulder_main(q, y)
    c_main = float(q_guess) if q_guess is not None else float(q[i_main])
    if shoulder_mode:
        c_sh = float(q[i_sh])
        mask = (q > c_sh - 0.03) & (q < c_main + 0.015) & np.isfinite(y)
    elif q_guess is not None:
        mask = np.isfinite(y)
    else:
        mask = (q > c_main - 0.06) & (q < c_main + 0.06) & np.isfinite(y)
    q_fit, y_fit = q[mask], y[mask]
    p0 = [max(y_fit.max() - y_fit.min(), 0.05), c_main, 0.01, 0.5, float(np.percentile(y_fit, 10))]
    if shoulder_mode:
        sigma = np.ones_like(y_fit) * 0.15
        sigma[q_fit < c_main - 0.001] = 0.04
        bounds = ([0.0, c_main - 0.015, 1e-4, 0.0, 0.0], [2.0, c_main + 0.015, 0.03, 1.0, 0.12])
        popt, pcov = curve_fit(
            pseudo_voigt, q_fit, y_fit, p0=p0, bounds=bounds,
            sigma=sigma, absolute_sigma=True, maxfev=20000,
        )
    else:
        dq = 0.012 if q_guess is not None else 0.05
        fwhm_max = 0.05 if q_guess is not None else 0.2
        bounds = ([0.0, c_main - dq, 1e-4, 0.0, -np.inf], [3.0, c_main + dq, fwhm_max, 1.0, np.inf])
        popt, pcov = curve_fit(pseudo_voigt, q_fit, y_fit, p0=p0, bounds=bounds, maxfev=20000)
    perr = np.sqrt(np.maximum(np.diag(pcov), 0.0))
    q_fine = np.linspace(q_fit.min(), q_fit.max(), 400)
    y_full = pseudo_voigt(q_fine, *popt)
    y_peak = y_full - popt[4]
    peak_h = float(np.max(y_peak))
    fwhm = float(popt[2])
    fwhm_err = float(perr[2])
    fwhm_max = 0.05 if q_guess is not None else 0.2
    if (not shoulder_mode) and (fwhm > 0.9 * fwhm_max or fwhm_err > 0.02 or fwhm <= 0):
        y_hm = y_fit - np.percentile(y_fit, 15)
        if y_hm.max() > 0:
            y_hm = y_hm / y_hm.max()
            above = np.where(y_hm >= 0.5)[0]
            if len(above) >= 3:
                fwhm = abs(float(q_fit[above[-1]] - q_fit[above[0]]))
                fwhm_err = max(0.25 * fwhm, 0.002)
    xi_nm = 2.0 * np.pi * SCHERRER_K / fwhm * 0.1
    xi_err = abs(2.0 * np.pi * SCHERRER_K / fwhm ** 2 * fwhm_err) * 0.1
    return {
        'q_data': q_fit,
        'y_data': (y_fit - popt[4]) / peak_h,
        'q_fine': q_fine,
        'y_fit': y_peak / peak_h,
        'center': float(popt[1]),
        'center_err': float(perr[1]),
        'fwhm': float(fwhm),
        'fwhm_err': float(fwhm_err),
        'xi_nm': float(xi_nm),
        'xi_err_nm': float(xi_err),
    }


In [ ]:
records = []
for T, scan, hkl in ALL_PEAK_SCANS:
    processed = find_processed(T, scan)
    use_processed = processed is not None and hkl in ('120', '111')
    if use_processed:
        rec = load_processed(processed)
    else:
        rec = load_raw_roi(T, scan, hkl)
    rec['hkl'] = hkl
    rec['scan'] = scan
    records.append(rec)
    print(f"{T:g} K  ({hkl:5s})  {rec['source']:10s}  n={len(rec['intensity'])}  {rec['path'].name}")

ref = next(r for r in records if abs(r['temperature'] - 320) < 1 and r['hkl'] == '120')
peak_global = ref['crop_v0'] + int(np.argmax(ref['intensity']))


def q_at(ver0):
    return qy_vertical_rows(
        np.array([peak_global]), ver0, L_TRUE_MM, ref['wl'], ref['tth'], ref['pix']
    )[0]


ver0_cal = float(brentq(lambda v: q_at(v) - Q_REF_120, 0.0, 2500.0))
print(f"Q calibration: L={L_TRUE_MM:.0f} mm, (120) at {Q_REF_120:.4f} A^-1, ver0={ver0_cal:.1f} px")

for rec in records:
    rows = rec['crop_v0'] + np.arange(len(rec['intensity']))
    q = qy_vertical_rows(rows, ver0_cal, L_TRUE_MM, rec['wl'], rec['tth'], rec['pix'])
    intensity = rec['intensity'].astype(float)
    q_guess = None
    if rec['source'] == 'raw' and rec['hkl'] not in ('120', '111'):
        row0 = rec.get('peak_row', rec.get('seed_row'))
        q_guess = float(qy_vertical_rows(
            np.array([row0]), ver0_cal, L_TRUE_MM, rec['wl'], rec['tth'], rec['pix']
        )[0])
        near = np.abs(rows - row0) <= 40
        if np.count_nonzero(near) >= 8:
            q = q[near]
            intensity = intensity[near]
    rec['q'] = q
    rec['intensity'] = intensity
    T = rec['temperature']
    rec['fit'] = fit_brightest(
        rec['q'], rec['intensity'],
        shoulder_mode=(rec['hkl'] in ('120', '111') and (T <= 330 or abs(T - 380) < 2)),
        q_guess=q_guess,
    )
    f = rec['fit']
    print(
        f"T={T:.0f} K  ({rec['hkl']:5s})  Q0={f['center']:.4f}±{f['center_err']:.4f}  "
        f"xi={f['xi_nm']:.1f}±{f['xi_err_nm']:.1f} nm"
    )


In [ ]:
temps = sorted({int(round(r['temperature'])) for r in records})
by_T = {T: [r for r in records if int(round(r['temperature'])) == T] for T in temps}

fig, (ax_w, ax_t) = plt.subplots(1, 2, figsize=FIGSIZE)

for i, T in enumerate(temps):
    off = i * OFFSET
    z = i + 1
    for rec in by_T[T]:
        color = HKL_COLORS[rec['hkl']]
        fit = rec['fit']
        ax_w.fill_between(fit['q_fine'], off, off + fit['y_fit'], color=color, alpha=0.28, zorder=2 * z, lw=0)
        ax_w.plot(fit['q_data'], off + fit['y_data'], color=color, lw=1.3, zorder=2 * z + 1)
    ax_w.text(1.048, off + 0.15, f'{T:g} K', ha='right', va='bottom', fontsize=8,
              color=temperature_color(T, cmap=cmap, norm=norm))

handles = [
    plt.Line2D([0], [0], color=HKL_COLORS[h], lw=1.6, label=HKL_LABELS[h])
    for h in ('011', '10-1', '120', '101', '111')
]
ax_w.legend(handles=handles, loc='upper right', handlelength=1.2)
ax_w.set_xlabel(r'$|\mathbf{Q}|$ ($\mathrm{\AA}^{-1}$)')
ax_w.set_ylabel('I (arb.), offset by T')
ax_w.set_xlim(1.05, 1.175)
ax_w.set_ylim(-0.15, (len(temps) - 1) * OFFSET + 1.2)
ax_w.set_yticks([])
ax_w.spines['top'].set_visible(False)
ax_w.spines['right'].set_visible(False)
ax_w.tick_params(top=False, right=False)

ax_xi = ax_t.twinx()
ax_t.set_xlim(312, 428)
ax_t.set_ylim(1.065, 1.175)
ax_xi.set_ylim(0, 120)
ax_t.imshow(
    np.linspace(0, 1, 512).reshape(1, -1),
    aspect='auto',
    cmap=spt_cmap,
    extent=[SPT[0], SPT[1], *ax_t.get_ylim()],
    origin='lower',
    interpolation='bicubic',
    zorder=0,
)
ax_t.set_xlim(312, 428)

for hkl in ('011', '10-1', '120', '101', '111'):
    rows = [r for r in records if r['hkl'] == hkl]
    if not rows:
        continue
    rows.sort(key=lambda r: r['temperature'])
    T = np.array([r['temperature'] for r in rows])
    q0 = np.array([r['fit']['center'] for r in rows])
    qe = np.array([r['fit']['center_err'] for r in rows])
    xi = np.array([r['fit']['xi_nm'] for r in rows])
    xe = np.array([r['fit']['xi_err_nm'] for r in rows])
    color = HKL_COLORS[hkl]
    marker = MARKERS[hkl]
    msize = 9 if hkl == '111' else 6
    qe_plot = np.where(np.isfinite(qe) & (qe < 0.01), qe, 0.0)
    xe_plot = np.where(np.isfinite(xe) & (xe < 25.0), xe, 0.0)
    ax_t.errorbar(
        T, q0, yerr=qe_plot, fmt=marker + '-', color=color, ecolor=color,
        elinewidth=0.9, capsize=2.5, capthick=0.8, markersize=msize,
        markerfacecolor=color, markeredgecolor=color, lw=1.2, zorder=4,
        label=HKL_LABELS[hkl],
    )
    ax_xi.errorbar(
        T, xi, yerr=xe_plot, fmt=marker + '--', color=color, ecolor=color,
        elinewidth=0.9, capsize=2.5, capthick=0.8, markersize=msize,
        markerfacecolor='white', markeredgecolor=color, lw=1.2, zorder=3,
    )

ax_t.set_xlabel('T (K)')
ax_t.set_ylabel(r'$|\mathbf{Q}_0|$ ($\mathrm{\AA}^{-1}$)', color=blue)
ax_xi.set_ylabel(r'$\xi$ (nm)', color=red)
ax_t.set_xticks([320, 340, 360, 380, 400, 420])
ax_t.tick_params(axis='y', colors=blue)
ax_xi.tick_params(axis='y', colors=red)
ax_t.spines['left'].set_color(blue)
ax_xi.spines['right'].set_color(red)
ax_t.spines['right'].set_visible(False)
ax_xi.spines['left'].set_visible(False)
ax_t.legend(loc='lower left', handlelength=1.4)

fig.tight_layout(w_pad=2.0)
out = Path('../figures')
out.mkdir(exist_ok=True)
fig.savefig(out / 'bragg_peaks_all.pdf')
fig.savefig(out / 'bragg_peaks_all.png')
print('saved', out / 'bragg_peaks_all.pdf')


In [ ]:
print(f"{'T (K)':>6}  {'hkl':>6}  {'|Q0|':>10}  {'±':>8}  {'ξ (nm)':>8}  {'±':>8}  {'FWHM':>8}  {'±':>8}")
print('-' * 74)
fit_path = Path('../figures/bragg_peaks_all_fits.csv')
fit_path.parent.mkdir(exist_ok=True)
lines = ['T_K,hkl,Q0_Ainv,Q0_err,xi_nm,xi_err,FWHM_Ainv,FWHM_err,source']
for rec in sorted(records, key=lambda r: (r['temperature'], r['hkl'])):
    f = rec['fit']
    T = rec['temperature']
    print(
        f"{T:6.0f}  {rec['hkl']:>6}  {f['center']:10.5f}  {f['center_err']:8.5f}  "
        f"{f['xi_nm']:8.2f}  {f['xi_err_nm']:8.2f}  {f['fwhm']:8.5f}  {f['fwhm_err']:8.5f}"
    )
    lines.append(
        f"{T:.0f},{rec['hkl']},{f['center']:.6f},{f['center_err']:.6f},"
        f"{f['xi_nm']:.4f},{f['xi_err_nm']:.4f},{f['fwhm']:.6f},{f['fwhm_err']:.6f},{rec['source']}"
    )
fit_path.write_text('\n'.join(lines) + '\n')
print('saved', fit_path)
